In [22]:
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Border
import openpyxl
import shutil
import os
import sys
from copy import copy
from openpyxl.utils import get_column_letter


def main():
    path = "./Database.xlsx"
    wb = load_workbook(path, data_only=True)
    # names = ["住民台帳", "水道料金", "電気料金", "ガス料金"]
    ws_jumin = wb["住民台帳"]

    ##### 1. Database.xlsxの内容の整合性確認 #####
    # Database.xlsx内のいずれかのシートで1行目から最終行目までの間で
    # 一つでも空白のセルがあった場合、エラー処理をする
    try:
        for ws in wb:
            for row in ws.iter_rows(values_only=True):
                # 各行でループ処理中、セルにNoneがあった場合はエラー
                if None in row:
                    raise ValueError(f"「{ws.title}」シートが不正です。Database.xlsxを再確認してください。")
    except ValueError as e:
        print(e)
        sys.exit(1)

    ##### 住民の氏名と住民票コードを取得しj_name、j_codeリストに入れる #####
    j_name = []
    j_code = []

    for row in ws_jumin.iter_rows(min_row=2, values_only=True):
        j_name.append(row[0])
        j_code.append(row[1])
        # print(j_code)


    ##### 出力するエクセル名「姓_名.xlsx」を作成 #####
    out_name = []
    for name in j_name:
        sei, mei = name.split()
        out_name.append(sei + "_" + mei + ".xlsx")
        # print(out_name)

    ##### ここで全住民の全シートを作成する #####
    # 住民票コードの数(住民数)だけループする
    for i in range(len(j_code)):
        ### 新しいワークブックを作成、最初からできるアクティブなシートを削除 ###
        new_book = Workbook()
        new_book.remove(new_book.active)
        # Database.xlsxの全シートをループする
        for sheet in wb:
            # Database.xlsxと同名のシートを新しいワークブックに作成
            new_sheet = new_book.create_sheet(sheet.title)
            # 列の幅をコピー（A, B, C… ごと）
            for col, dim in sheet.column_dimensions.items():
                for c in range(dim.min, dim.max + 1):
                    new_sheet.column_dimensions[get_column_letter(c)].width = dim.width
            # headerはDatabase.xlsxと同じ内容をコピー
            for row in sheet.iter_rows(min_row=1, max_row=1, values_only=True):
                new_sheet.append(row)
                # ターゲットの住民票コードをDatabase.xlsxの住民票コードと比較し、一致したら一行追加
                for row in sheet.iter_rows(min_row=2,values_only=True):
                    # row[1]:B列(住民票コード)
                    if j_code[i] == row[1]:
                        new_sheet.append(row)

                # シート名が"住民台帳"意外だった場合はA,B列の3行目以降を削除
                if sheet.title != "住民台帳":
                    for row in range(3, new_sheet.max_row + 1):
                        for col in range(1, 3):
                            new_sheet.cell(row=row, column=col).value = None

        new_book.save(out_name[i])
        print(f"{out_name[i]}の作成と書き込みが完了しました。")


# このファイルを直接実行したときだけ main() を呼び出す、というお決まりの書き方
if __name__ == "__main__":
    main()



佐藤_滋.xlsxの作成と書き込みが完了しました。
鈴木_健一.xlsxの作成と書き込みが完了しました。
田中_芳子.xlsxの作成と書き込みが完了しました。
田上_良太.xlsxの作成と書き込みが完了しました。
竹本_佳代子.xlsxの作成と書き込みが完了しました。
大元_久.xlsxの作成と書き込みが完了しました。
